In [2]:
import pandas as pd

features = [
    "Radius", "Texture", "Perimeter", "Area", "Smoothness", "Compactness",
    "Concavity", "Concave Points", "Symmetry", "Fractal Dimension",
]
features = [f"{feature}_{stat}" for feature in features for stat in ["mean", "se", "extreme"]]

df = pd.read_csv(
    "../data/data.csv",
    skiprows=1,
    header=None,
    names=["ID", "Diagnosis"] + features,
 )

df.head(3)

,ID,Diagnosis,Radius_mean,Radius_se,Radius_extreme,Texture_mean,Texture_se,Texture_extreme,Perimeter_mean,Perimeter_se,...,Concavity_extreme,Concave Points_mean,Concave Points_se,Concave Points_extreme,Symmetry_mean,Symmetry_se,Symmetry_extreme,Fractal Dimension_mean,Fractal Dimension_se,Fractal Dimension_extreme
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


In [23]:
df.describe()

,ID,Diagnosis,Radius,Texture,Perimeter,Area,Smoothness,Compactness,Concavity,Concave Points,Symmetry,Fractal Dimention
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,0.020542,0.003795,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,0.008266,0.002646,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,0.007882,0.000895,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,0.015160,0.002248,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,0.018730,0.003187,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,0.023480,0.004558,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,0.078950,0.029840,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import numpy as np
import tensorflow as tf

# Baseline NN matching the requested architecture and training setup.
data = df.copy()

if "ID" in data.columns:
    data = data.drop(columns=["ID"])

if "Diagnosis" in data.columns:
    y_raw = data["Diagnosis"].map({"M": 1, "B": 0}).to_numpy()
    X = data.drop(columns=["Diagnosis"]).to_numpy()
else:
    X = data.iloc[:, :-1].to_numpy()
    y_raw = data.iloc[:, -1].to_numpy()

# Split into train (70%), validation (15%), and test (15%).
X_train, X_temp, y_train_raw, y_temp_raw = train_test_split(
    X, y_raw, test_size=0.30, random_state=42, stratify=y_raw
)
X_val, X_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp_raw, test_size=0.50, random_state=42, stratify=y_temp_raw
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

num_classes = len(np.unique(y_raw))
y_train = tf.keras.utils.to_categorical(y_train_raw, num_classes=num_classes)
y_val = tf.keras.utils.to_categorical(y_val_raw, num_classes=num_classes)
y_test = tf.keras.utils.to_categorical(y_test_raw, num_classes=num_classes)

input_shape = X_train.shape[1]
output_shape = num_classes

# Equivalent to: model.createNetwork([...DenseLayer(...), ..., DenseLayer(output, softmax)...])
network = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(input_shape,)),
        tf.keras.layers.Dense(24, activation="sigmoid", kernel_initializer="he_uniform"),
        tf.keras.layers.Dense(24, activation="sigmoid", kernel_initializer="he_uniform"),
        tf.keras.layers.Dense(24, activation="sigmoid", kernel_initializer="he_uniform"),
        tf.keras.layers.Dense(output_shape, activation="softmax", kernel_initializer="he_uniform"),
    ]
)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0314)
network.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history = network.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    batch_size=8,
    epochs=84,
    verbose=0,
)

_, val_acc = network.evaluate(X_val, y_val, verbose=0)
_, test_acc = network.evaluate(X_test, y_test, verbose=0)
y_test_pred = np.argmax(network.predict(X_test, verbose=0), axis=1)

print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}, Test size: {len(X_test)}")
print(f"Validation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print("\nTest Classification Report:")
print(classification_report(y_test_raw, y_test_pred))

Train size: 398, Validation size: 85, Test size: 86
Validation Accuracy: 0.9647
Test Accuracy: 0.9884

Test Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        54
           1       1.00      0.97      0.98        32

    accuracy                           0.99        86
   macro avg       0.99      0.98      0.99        86
weighted avg       0.99      0.99      0.99        86

